In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path.home() / 'Downloads' / 'idx exchange' / 'csv'
MONTHS = pd.period_range('2024-01', '2026-04', freq='M')

print('Data folder:', DATA_DIR)
print('Folder exists:', DATA_DIR.exists())
print('Number of months:', len(MONTHS))

Data folder: /Users/jasmineyuan/Downloads/idx exchange/csv
Folder exists: True
Number of months: 28


In [2]:
# Select monthly CRMLS Sold files, excluding filled versions.

from pathlib import Path
import pandas as pd

# Set the data folder.
DATA_DIR = Path.home() / "Downloads" / "idx exchange" / "csv"

print("Data folder:", DATA_DIR)
print("Folder exists:", DATA_DIR.exists())

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Data folder not found: {DATA_DIR}")

# Select all CRMLS Sold CSV files.

selected_files = sorted(DATA_DIR.glob("CRMLSSold*.csv"))

# Create a list to store the monthly DataFrames.
frames = []

# Read each selected CSV file.
for path in selected_files:
    monthly_df = pd.read_csv(
        path,
        low_memory=False,
        dtype={"PostalCode": "string"},
    )

    # Add the source file month.
   
    monthly_df["source_month"] = (
    path.stem.removeprefix("CRMLSSold").removesuffix("_filled")
)

    # Store the monthly DataFrame.
    frames.append(monthly_df)

# Stop if no matching files were loaded.
if not frames:
    raise ValueError("No matching CSV files were found.")

# Combine all monthly DataFrames.
df = pd.concat(frames, ignore_index=True)

# Display the loading results.
print("Number of files loaded:", len(frames))
print("Combined dataset shape (rows, columns):", df.shape)

display(
    df.groupby("source_month")
    .size()
    .rename("Row count")
)

Data folder: /Users/jasmineyuan/Downloads/idx exchange/csv
Folder exists: True
Number of files loaded: 28
Combined dataset shape (rows, columns): (615707, 85)


source_month
202401    17958
202402    19925
202403    23276
202404    24640
202405    26487
202406    24328
202407    26240
202408    24558
202409    21267
202410    23274
202411    20279
202412    20241
202501    18738
202502    18702
202503    21445
202504    23262
202505    23154
202506    22883
202507    23646
202508    22972
202509    22443
202510    23233
202511    19088
202512    20538
202601    16487
202602    19010
202603    23372
202604    24261
Name: Row count, dtype: int64

In [3]:
# *only use information that’s available for any address at the time of prediction.
# * Property characteristics (square footage, beds, baths, lot size, age, location, school district, comp-based neighborhood prices) pass that test.


In [4]:
# explicity select the property feastures 

In [5]:

# Keep California single-family residential properties.
single_family_df = df.loc[
    df["PropertyType"].eq("Residential")
    & df["PropertySubType"].eq("SingleFamilyResidence")
    & df["StateOrProvince"].eq("CA")
].copy()

print("California single-family rows:", len(single_family_df))

California single-family rows: 309713


In [6]:
# Correct two sale prices using listing IDs and sale dates.
price_corrections = [
    ("NDP2601657", "2026-04-22", 645000000, 645000),
    ("NDP2511562", "2026-04-27", 796000000, 796000),
]

correction_log = []

for listing_id, close_date, old_price, new_price in price_corrections:
    mask = (
        single_family_df["ListingId"].astype("string").eq(listing_id)
        & pd.to_datetime(
            single_family_df["CloseDate"], errors="coerce"
        ).dt.normalize().eq(pd.Timestamp(close_date))
    )

    # Require exactly one matching record.
    assert mask.sum() == 1, (
        f"Expected one record for {listing_id}; found {mask.sum()}."
    )

    current_price = pd.to_numeric(
        single_family_df.loc[mask, "ClosePrice"],
        errors="raise",
    ).iloc[0]

    # Allow this cell to run again after correction.
    assert current_price in (old_price, new_price), (
        f"Unexpected ClosePrice for {listing_id}: {current_price}"
    )

    single_family_df.loc[mask, "ClosePrice"] = new_price

    correction_log.append({
        "ListingId": listing_id,
        "CloseDate": close_date,
        "OriginalClosePrice": old_price,
        "CorrectedClosePrice": new_price,
    })

display(pd.DataFrame(correction_log))

,ListingId,CloseDate,OriginalClosePrice,CorrectedClosePrice
0,NDP2601657,2026-04-22,645000000,645000
1,NDP2511562,2026-04-27,796000000,796000


In [7]:
# Audit sale prices against listing prices without changing the dataset.
audit_columns = [
    "source_month",
    "ListingKey",
    "ListingId",
    "CloseDate",
    "UnparsedAddress",
    "City",
    "PostalCode",
    "LivingArea",
    "OriginalListPrice",
    "ListPrice",
    "ClosePrice",
]

price_audit = single_family_df[audit_columns].copy()

# Preserve the original row index for tracing records.
price_audit.insert(0, "SourceRowIndex", price_audit.index)

# Parse prices in the audit copy only.
for col in ["ClosePrice", "ListPrice", "OriginalListPrice"]:
    price_audit[col] = pd.to_numeric(
        price_audit[col]
        .astype("string")
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.strip(),
        errors="coerce",
    )

# Identify positive, finite sale and listing prices.
valid_close = (
    price_audit["ClosePrice"].notna()
    & np.isfinite(price_audit["ClosePrice"])
    & price_audit["ClosePrice"].gt(0)
)

valid_list = (
    price_audit["ListPrice"].notna()
    & np.isfinite(price_audit["ListPrice"])
    & price_audit["ListPrice"].gt(0)
)

valid_pair = valid_close & valid_list

# Calculate ratios only where both prices are valid.
price_audit["CloseToListRatio"] = (
    price_audit["ClosePrice"].where(valid_pair)
    / price_audit["ListPrice"].where(valid_pair)
)

price_audit["AbsoluteDifference"] = (
    price_audit["ClosePrice"] - price_audit["ListPrice"]
).abs().where(valid_pair)

# A ratio of 1 is neutral.
# Ratios of 0.001 and 1000 receive the same anomaly score.
price_audit["RatioAnomalyScore"] = np.abs(
    np.log(price_audit["CloseToListRatio"])
)

# Use broad review thresholds, not automatic deletion rules.
LOWER_RATIO = 0.5
UPPER_RATIO = 2.0

price_audit["ReviewReason"] = ""

price_audit.loc[
    valid_pair & price_audit["CloseToListRatio"].lt(LOWER_RATIO),
    "ReviewReason",
] = "Sale price below 50% of listing price"

price_audit.loc[
    valid_pair & price_audit["CloseToListRatio"].gt(UPPER_RATIO),
    "ReviewReason",
] = "Sale price above 200% of listing price"

price_audit.loc[
    ~valid_list,
    "ReviewReason",
] = "Missing or invalid listing price"

price_audit.loc[
    ~valid_close,
    "ReviewReason",
] = "Missing or invalid sale price"

# Keep all ratio candidates, ranked from most to least unusual.
ratio_candidates = (
    price_audit.loc[
        valid_pair
        & (
            price_audit["CloseToListRatio"].lt(LOWER_RATIO)
            | price_audit["CloseToListRatio"].gt(UPPER_RATIO)
        )
    ]
    .sort_values("RatioAnomalyScore", ascending=False)
    .copy()
)

# Keep records that cannot be checked using a price ratio separately.
invalid_price_records = price_audit.loc[~valid_pair].copy()

# Summarize coverage and review counts.
display(pd.DataFrame({
    "Check": [
        "Total residential single-family records",
        "Records with valid sale and listing prices",
        "Ratio candidates requiring review",
        "Records with missing or invalid prices",
    ],
    "Count": [
        len(price_audit),
        int(valid_pair.sum()),
        len(ratio_candidates),
        len(invalid_price_records),
    ],
}))

# Display every flagged ratio candidate.
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
):
    display(ratio_candidates)

    if not invalid_price_records.empty:
        display(invalid_price_records)

/Users/jasmineyuan/idx-jupyter-env/lib/python3.13/site-packages/pandas/core/arrays/masked.py:788: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs2, **kwargs)


,Check,Count
0,Total residential single-family records,309713
1,Records with valid sale and listing prices,309710
2,Ratio candidates requiring review,232
3,Records with missing or invalid prices,3


,SourceRowIndex,source_month,ListingKey,ListingId,CloseDate,UnparsedAddress,City,PostalCode,LivingArea,OriginalListPrice,ListPrice,ClosePrice,CloseToListRatio,AbsoluteDifference,RatioAnomalyScore,ReviewReason
270587,270587,202412,1077504198,224002893,2024-12-09,7013 Green Vista Circle,West Hills,91307,1811.0,1249000.0,1199000.0,1.15,0.000001,1198998.85,13.857236,Sale price below 50% of listing price
543168,543168,202601,1144223437,219137367DA,2026-01-30,600 Gold Canyon Drive,Palm Desert,92211,3513.0,1795000.0,1749000.0,1.75,0.000001,1748998.25,13.814939,Sale price below 50% of listing price
108697,108697,202405,1059929060,41049105,2024-05-15,1825 Cresthaven Street,Brentwood,94513,2048.0,784345.0,784345.0,345.0,0.00044,784000.0,7.72906,Sale price below 50% of listing price
167195,167195,202408,1077754080,41067360,2024-08-14,168 Norman ave,NaN,94520-1105,1047.0,325000.0,325000.0,375000000.0,1153.846154,374675000.0,7.050856,Sale price above 200% of listing price
448736,448736,202509,1131758129,PTP2506723,2025-09-26,7280 Cornell Avenue,La Mesa,91942,798.0,699000.0,699000.0,751412000.0,1074.981402,750713000.0,6.980059,Sale price above 200% of listing price
480048,480048,202510,1129501416,219134383PS,2025-10-23,68110 Risueno Road,Cathedral City,92234,1305.0,498000.0,498000.0,485.0,0.000974,497515.0,6.934206,Sale price below 50% of listing price
610980,610980,202604,1150712418,PTP2600266,2026-04-09,3373 Soldau Drive,San Diego,92154,1444.0,750000.0,750000.0,768500000.0,1024.666667,767750000.0,6.932123,Sale price above 200% of listing price
475560,475560,202510,1133720285,PW25214302,2025-10-31,25902 Matel Road,Valencia,91355,2289.0,979000.0,979000.0,989500000.0,1010.72523,988521000.0,6.918423,Sale price above 200% of listing price
464634,464634,202509,1115065523,NDP2505639,2025-09-10,21905 Grovepark Drive,Saugus,91350,1900.0,839900.0,809000.0,815000000.0,1007.416564,814191000.0,6.915144,Sale price above 200% of listing price
507181,507181,202511,1126046881,PTP2506039,2025-11-24,1793 Thelborn Way,San Diego,92154,1271.0,750000.0,735000.0,740000000.0,1006.802721,739265000.0,6.914535,Sale price above 200% of listing price


,SourceRowIndex,source_month,ListingKey,ListingId,CloseDate,UnparsedAddress,City,PostalCode,LivingArea,OriginalListPrice,ListPrice,ClosePrice,CloseToListRatio,AbsoluteDifference,RatioAnomalyScore,ReviewReason
136081,136081,202406,1045745288,MP23181752,2024-06-06,41147 Marble Ct,Oakhurst,93644,2200.0,575000.0,575000.0,<NA>,<NA>,<NA>,<NA>,Missing or invalid sale price
267022,267022,202412,1086646611,MC24191255,2024-12-13,18806 S Nancy Emilia Court,Tracy,95304,3109.0,2500.0,2700000.0,<NA>,<NA>,<NA>,<NA>,Missing or invalid sale price
407174,407174,202507,1117019091,P1-22708,2025-07-30,5026 Doreen Avenue,Temple City,91780,2001.0,1350000.0,1350000.0,0.0,<NA>,<NA>,<NA>,Missing or invalid sale price


In [8]:
from pathlib import Path

# Save the current ratio candidates before applying corrections.
AUDIT_DIR = (
    Path.home()
    / "Downloads"
    / "IDX-Exchange-Data-Science"
    / "price_audit"
)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# Preserve the initial audit when this cell is rerun.
audit_file = AUDIT_DIR / "closeprice_ratio_candidates_initial.csv"

if not audit_file.exists():
    ratio_candidates.to_csv(audit_file, index=False)

# Apply individually verified sale-price corrections.
# Listing prices are used for inspection only, not as replacement targets.
verified_corrections = pd.DataFrame(
    [
        ("224002893",   "2024-12-09",         1.15, 1150000),
        ("219137367DA", "2026-01-30",         1.75, 1750000),
        ("41049105",    "2024-05-15",       345.00,  784345),
        ("219113154PS", "2024-09-03",       380.00,  380000),
        ("219134383PS", "2025-10-23",       485.00,  485000),
        ("IG25124853",  "2025-07-31",     15000.00,  433000),
        ("41067360",    "2024-08-14", 375000000.00,  375000),
        ("41094404",    "2025-05-30", 600000000.00,  600000),
        ("PTP2506039",  "2025-11-24", 740000000.00,  740000),
        ("PTP2506723",  "2025-09-26", 751412000.00,  715412),
        ("NDP2505639",  "2025-09-10", 815000000.00,  815000),
        ("PTP2507350",  "2025-09-25", 875000000.00,  875000),
        ("NDP2500057",  "2025-04-11", 970000000.00,  970000),
        ("PW25214302",  "2025-10-31", 989500000.00,  989500),
    ],
    columns=[
        "ListingId",
        "CloseDate",
        "OriginalClosePrice",
        "CorrectedClosePrice",
    ],
)

listing_ids = (
    single_family_df["ListingId"]
    .astype("string")
    .str.strip()
)

close_dates = pd.to_datetime(
    single_family_df["CloseDate"],
    errors="coerce",
).dt.normalize()

close_prices = pd.to_numeric(
    single_family_df["ClosePrice"],
    errors="coerce",
)

# Validate every correction before changing any values.
correction_masks = []

for row in verified_corrections.itertuples(index=False):
    mask = (
        listing_ids.eq(row.ListingId)
        & close_dates.eq(pd.Timestamp(row.CloseDate))
    ).fillna(False)

    if not mask.any():
        raise ValueError(
            f"Record not found: {row.ListingId}, {row.CloseDate}"
        )

    # Accept the original value or the corrected value when rerunning.
    expected_price = close_prices.loc[mask].isin([
        row.OriginalClosePrice,
        row.CorrectedClosePrice,
    ])

    if not expected_price.all():
        raise ValueError(
            f"Unexpected ClosePrice: {row.ListingId}, {row.CloseDate}"
        )

    correction_masks.append(mask)

# Update the working dataset and record the action.
training_price_correction_log = []

for row, mask in zip(
    verified_corrections.itertuples(index=False),
    correction_masks,
):
    changed_rows = int(
        close_prices.loc[mask].ne(row.CorrectedClosePrice).sum()
    )

    single_family_df.loc[mask, "ClosePrice"] = (
        row.CorrectedClosePrice
    )

    training_price_correction_log.append({
        "ListingId": row.ListingId,
        "CloseDate": row.CloseDate,
        "OriginalClosePrice": row.OriginalClosePrice,
        "CorrectedClosePrice": row.CorrectedClosePrice,
        "MatchedRows": int(mask.sum()),
        "ChangedRows": changed_rows,
    })

training_price_correction_log = pd.DataFrame(
    training_price_correction_log
)

training_price_correction_log.to_csv(
    AUDIT_DIR / "verified_training_price_corrections.csv",
    index=False,
)

display(training_price_correction_log)

print(
    "Rows corrected:",
    training_price_correction_log["ChangedRows"].sum(),
)
print("Initial ratio audit saved to:", audit_file)

,ListingId,CloseDate,OriginalClosePrice,CorrectedClosePrice,MatchedRows,ChangedRows
0,224002893,2024-12-09,1.150000e+00,1150000,1,1
1,219137367DA,2026-01-30,1.750000e+00,1750000,1,1
2,41049105,2024-05-15,3.450000e+02,784345,1,1
3,219113154PS,2024-09-03,3.800000e+02,380000,1,1
4,219134383PS,2025-10-23,4.850000e+02,485000,1,1
5,IG25124853,2025-07-31,1.500000e+04,433000,1,1
6,41067360,2024-08-14,3.750000e+08,375000,1,1
7,41094404,2025-05-30,6.000000e+08,600000,1,1
8,PTP2506039,2025-11-24,7.400000e+08,740000,1,1
9,PTP2506723,2025-09-26,7.514120e+08,715412,1,1


Rows corrected: 14
Initial ratio audit saved to: /Users/jasmineyuan/Downloads/IDX-Exchange-Data-Science/price_audit/closeprice_ratio_candidates_initial.csv


In [9]:
# Fields that depend on a listing or transaction.
excluded_columns = [
    # Listing prices and market duration
    "OriginalListPrice",
    "ListPrice",
    "DaysOnMarket",

    # Listing identifiers and status
    "ListingKey",
    "ListingKeyNumeric",
    "ListingId",
    "MlsStatus",

    # Listing and transaction dates
    "ContractStatusChangeDate",
    "PurchaseContractDate",
    "ListingContractDate",

    # Listing agents and offices
    "ListAgentEmail",
    "ListAgentFirstName",
    "ListAgentLastName",
    "ListAgentFullName",
    "ListOfficeName",
    "CoListOfficeName",
    "CoListAgentFirstName",
    "CoListAgentLastName",

    # Buyer agents and offices
    "BuyerOfficeName",
    "BuyerOfficeAOR",
    "BuyerAgentMlsId",
    "BuyerAgentFirstName",
    "BuyerAgentLastName",
    "CoBuyerAgentFirstName",

    # Transaction compensation
    "BuyerAgencyCompensationType",
    "BuyerAgencyCompensation",
    
    # Agent affiliations and MLS source metadata
    "BuyerAgentAOR",
    "ListAgentAOR",
    "OriginatingSystemName",
    "OriginatingSystemSubName",

]

# Keep the target and sale dates separately.
y = single_family_df["ClosePrice"].copy()
sale_dates = single_family_df["CloseDate"].copy()

# Keep source metadata outside the property audit.
metadata_columns = [
    "source_month",
    "source_file",
    "source_is_filled",
]

# Retain the remaining columns for missing-value inspection.
audit_columns = [
    col for col in single_family_df.columns
    if col not in excluded_columns
    + ["ClosePrice", "CloseDate"]
    + metadata_columns
]

property_data = single_family_df[audit_columns].copy()

# Convert blank strings to missing values.
property_data = property_data.replace(r"^\s*$", pd.NA, regex=True)

# Summarize missing values.
missing_summary = pd.DataFrame({
    "Data type": property_data.dtypes.astype(str),
    "Missing count": property_data.isna().sum(),
    "Missing percentage": (
        property_data.isna().mean() * 100
    ).round(2),
}).sort_values("Missing percentage", ascending=False)

print("Total single-family rows:", len(property_data))
print("Number of retained columns:", property_data.shape[1])

display(missing_summary)

# Inspect missing target values separately.
print("Missing ClosePrice count:", y.isna().sum())

Total single-family rows: 309713
Number of retained columns: 52


,Data type,Missing count,Missing percentage
TaxYear,float64,309713,100.00
TaxAnnualAmount,float64,309713,100.00
MiddleOrJuniorSchoolDistrict,float64,309713,100.00
BusinessType,str,309713,100.00
FireplacesTotal,float64,309713,100.00
ElementarySchoolDistrict,float64,309713,100.00
AboveGradeFinishedArea,float64,309713,100.00
CoveredSpaces,float64,309713,100.00
WaterfrontYN,object,309577,99.96
BelowGradeFinishedArea,float64,307811,99.39


Missing ClosePrice count: 2


In [10]:
# Identify columns with no observed values.
all_missing_columns = property_data.columns[
    property_data.isna().all()
].tolist()

print("Columns dropped because all values are missing:")
print(all_missing_columns)

# Drop empty columns while retaining all rows.
property_data = property_data.drop(columns=all_missing_columns)

print("Remaining columns:", property_data.shape[1])
print("Remaining rows:", len(property_data))

Columns dropped because all values are missing:
['FireplacesTotal', 'AboveGradeFinishedArea', 'TaxAnnualAmount', 'TaxYear', 'ElementarySchoolDistrict', 'BusinessType', 'CoveredSpaces', 'MiddleOrJuniorSchoolDistrict']
Remaining columns: 44
Remaining rows: 309713


In [11]:
# Inspect the values in the added columns.
for col in ["latfilled", "lonfilled"]:
    print(f"\nColumn: {col}")
    print(property_data[col].value_counts(dropna=False).head(10))


Column: latfilled
latfilled
NaN      229123
False     46782
True      33808
Name: count, dtype: int64

Column: lonfilled
lonfilled
NaN      229123
False     46782
True      33808
Name: count, dtype: int64


In [12]:
# Remove coordinate-fill flags.
property_data = property_data.drop(columns=["latfilled", "lonfilled"])

In [13]:
import numpy as np

# Convert the target to numeric values.
y = pd.to_numeric(y, errors="coerce")

# Identify valid close prices.
valid_target = y.notna() & np.isfinite(y) & y.gt(0)

print("Rows removed due to invalid ClosePrice:", (~valid_target).sum())

# Keep property data, target, and sale dates aligned.
property_data = property_data.loc[valid_target].copy()
y = y.loc[valid_target].copy()
sale_dates = sale_dates.loc[valid_target].copy()

print("Remaining rows:", len(property_data))
print("Missing ClosePrice count:", y.isna().sum())

Rows removed due to invalid ClosePrice: 3
Remaining rows: 309710
Missing ClosePrice count: 0


In [14]:
# Verify row counts and index alignment.
print("Row counts:", len(property_data), len(y), len(sale_dates))
print(
    "Indexes aligned:",
    property_data.index.equals(y.index)
    and property_data.index.equals(sale_dates.index)
)

Row counts: 309710 309710 309710
Indexes aligned: True


In [15]:
# Inspect columns with more than 90% missing values.
missing_rate = property_data.isna().mean()

high_missing_columns = missing_rate[
    missing_rate > 0.90
].index.tolist()

high_missing_summary = pd.DataFrame({
    "Missing percentage": (
        missing_rate[high_missing_columns] * 100
    ).round(2),
    "Non-missing count": (
        property_data[high_missing_columns].notna().sum()
    ),
}).sort_values("Missing percentage", ascending=False)

display(high_missing_summary)

# Inspect observed values before deciding how to handle each column.
for col in high_missing_columns:
    print(f"\nColumn: {col}")
    print(property_data[col].value_counts().head(10))

,Missing percentage,Non-missing count
WaterfrontYN,99.96,136
BelowGradeFinishedArea,99.39,1902
BasementYN,97.59,7454
BuilderName,95.33,14458
LotSizeDimensions,93.88,18960
BuildingAreaTotal,93.65,19655



Column: WaterfrontYN
WaterfrontYN
True    136
Name: count, dtype: int64

Column: BasementYN
BasementYN
True    7454
Name: count, dtype: int64

Column: BuilderName
BuilderName
Lennar               944
Richmond American    661
D.R. Horton          610
Meritage Homes       504
DR Horton            496
Toll Brothers        450
Taylor Morrison      430
Shea Homes           367
D. R. Horton         351
Pulte Homes          268
Name: count, dtype: int64

Column: BuildingAreaTotal
BuildingAreaTotal
0.0       94
1600.0    44
1200.0    40
1400.0    38
1800.0    38
1500.0    38
1680.0    38
2000.0    34
1260.0    32
4000.0    30
Name: count, dtype: int64

Column: BelowGradeFinishedArea
BelowGradeFinishedArea
0.0       1532
1.0        158
600.0       11
100.0       10
500.0        8
1000.0       6
200.0        5
800.0        5
400.0        4
300.0        4
Name: count, dtype: int64

Column: LotSizeDimensions
LotSizeDimensions
50x135    347
50x100    248
50x120    235
50x130    223
50x150    203
5

In [16]:
# keep to remove those properties

In [17]:
# Drop sparse columns from the initial feature dataset.
sparse_columns_to_drop = [
    "WaterfrontYN",
    "BasementYN",
    "BuilderName",
    "BuildingAreaTotal",
    "BelowGradeFinishedArea",
    "LotSizeDimensions",
]

property_data = property_data.drop(columns=sparse_columns_to_drop)

print("Remaining columns:", property_data.shape[1])
print("Remaining rows:", len(property_data))

Remaining columns: 36
Remaining rows: 309710


In [18]:
# Inspect the values in remaining Yes/No columns.
yn_columns = [
    col for col in property_data.columns
    if col.endswith("YN")
]

for col in yn_columns:
    print(f"\nColumn: {col}")
    print(property_data[col].value_counts(dropna=False))


Column: ViewYN
ViewYN
True     170254
False    111845
NaN       27611
Name: count, dtype: int64

Column: PoolPrivateYN
PoolPrivateYN
False    234250
True      45669
NaN       29791
Name: count, dtype: int64

Column: AttachedGarageYN
AttachedGarageYN
True     227955
False     45720
NaN       36035
Name: count, dtype: int64

Column: FireplaceYN
FireplaceYN
True     225055
False     84449
NaN         206
Name: count, dtype: int64

Column: NewConstructionYN
NewConstructionYN
False    276172
NaN       22487
True      11051
Name: count, dtype: int64


In [19]:
# Mark missing Yes/No values as "Unknown".
for col in yn_columns:
    property_data[col] = (
        property_data[col]
        .astype("string")
        .fillna("Unknown")
    )

# Verify the categories and counts.
for col in yn_columns:
    print(f"\nColumn: {col}")
    print(property_data[col].value_counts(dropna=False))

print("\nRemaining rows:", len(property_data))


Column: ViewYN
ViewYN
True       170254
False      111845
Unknown     27611
Name: count, dtype: Int64

Column: PoolPrivateYN
PoolPrivateYN
False      234250
True        45669
Unknown     29791
Name: count, dtype: Int64

Column: AttachedGarageYN
AttachedGarageYN
True       227955
False       45720
Unknown     36035
Name: count, dtype: Int64

Column: FireplaceYN
FireplaceYN
True       225055
False       84449
Unknown       206
Name: count, dtype: Int64

Column: NewConstructionYN
NewConstructionYN
False      276172
Unknown     22487
True        11051
Name: count, dtype: Int64

Remaining rows: 309710


In [20]:
# Find text columns other than the Yes/No fields.
text_columns = [
    col
    for col in property_data.select_dtypes(
        include=["object", "string"]
    ).columns
    if col not in yn_columns
]

# Inspect missing values in these columns.
text_missing_summary = pd.DataFrame({
    "Missing count": property_data[text_columns].isna().sum(),
    "Missing percentage": (
        property_data[text_columns].isna().mean() * 100
    ).round(2),
})

In [21]:
display(text_missing_summary)

,Missing count,Missing percentage
Flooring,110708,35.75
UnparsedAddress,294,0.09
PropertyType,0,0.00
AssociationFeeFrequency,231900,74.88
MLSAreaMajor,44008,14.21
CountyOrParish,0,0.00
ElementarySchool,267338,86.32
PropertySubType,0,0.00
SubdivisionName,202363,65.34
City,240,0.08


In [22]:
# Show remaining missing values across all retained columns.
remaining_missing = property_data.isna().sum()

display(
    remaining_missing[remaining_missing > 0]
    .sort_values(ascending=False)
    .rename("Missing count")
)

ElementarySchool           267338
MiddleOrJuniorSchool       267009
HighSchool                 254287
AssociationFeeFrequency    231900
SubdivisionName            202363
MainLevelBedrooms          124728
Flooring                   110708
AssociationFee              93563
HighSchoolDistrict          80644
MLSAreaMajor                44008
Stories                     39369
Levels                      29245
GarageSpaces                11517
LotSizeAcres                 5353
LotSizeSquareFeet            5326
LotSizeArea                  5302
Latitude                     2712
Longitude                    2712
StreetNumberNumeric           362
ParkingTotal                  348
UnparsedAddress               294
City                          240
YearBuilt                     223
LivingArea                    164
BathroomsTotalInteger          50
PostalCode                      1
Name: Missing count, dtype: int64

In [23]:
# Drop school-name fields with limited coverage.
school_columns_to_drop = [
    "ElementarySchool",
    "MiddleOrJuniorSchool",
    "HighSchool",
]

property_data = property_data.drop(columns=school_columns_to_drop)

print("Remaining columns:", property_data.shape[1])
print("Remaining rows:", len(property_data))

Remaining columns: 33
Remaining rows: 309710


In [24]:
# Mark missing categorical values as Unknown.
categorical_columns = [
    "Flooring",
    "MLSAreaMajor",
    "SubdivisionName",
    "Levels",
    "HighSchoolDistrict",
]

for col in categorical_columns:
    property_data[col] = (
        property_data[col]
        .astype("string")
        .fillna("Unknown")
    )

# Verify that missing values have been handled.
print(property_data[categorical_columns].isna().sum())
print("Remaining rows:", len(property_data))

Flooring              0
MLSAreaMajor          0
SubdivisionName       0
Levels                0
HighSchoolDistrict    0
dtype: int64
Remaining rows: 309710


In [25]:
# Mark missing categorical values as Unknown.
categorical_columns = [
    "Flooring",
    "MLSAreaMajor",
    "SubdivisionName",
    "Levels",
    "HighSchoolDistrict",
]

for col in categorical_columns:
    property_data[col] = (
        property_data[col]
        .astype("string")
        .fillna("Unknown")
    )

# Verify that missing values have been handled.
print(property_data[categorical_columns].isna().sum())
print("Remaining rows:", len(property_data))

Flooring              0
MLSAreaMajor          0
SubdivisionName       0
Levels                0
HighSchoolDistrict    0
dtype: int64
Remaining rows: 309710


In [26]:
# Inspect HOA payment-frequency categories.
print(property_data["AssociationFeeFrequency"].value_counts(dropna=False))

# Check whether HOA fee and frequency are missing together.
display(
    pd.crosstab(
        property_data["AssociationFee"].isna(),
        property_data["AssociationFeeFrequency"].isna(),
        rownames=["Fee is missing"],
        colnames=["Frequency is missing"],
    )
)

AssociationFeeFrequency
NaN             231900
Monthly          68213
Annually          6306
Quarterly         2883
SemiAnnually       408
Name: count, dtype: int64


Frequency is missing,False,True
Fee is missing,,
False,77757,138390
True,53,93510


In [27]:
# Compare missing HOA fees and payment frequencies.
# True：missing；False：have values recorded
display(
    pd.crosstab(
        property_data["AssociationFee"].isna(),
        property_data["AssociationFeeFrequency"].isna(),
        rownames=["Fee is missing"],
        colnames=["Frequency is missing"],
    )
)

Frequency is missing,False,True
Fee is missing,,
False,77757,138390
True,53,93510


In [28]:
# Mark missing payment frequencies as Unknown.
property_data["AssociationFeeFrequency"] = (
    property_data["AssociationFeeFrequency"]
    .astype("string")
    .fillna("Unknown")
)

print(
    property_data["AssociationFeeFrequency"]
    .value_counts(dropna=False)
)

AssociationFeeFrequency
Unknown         231900
Monthly          68213
Annually          6306
Quarterly         2883
SemiAnnually       408
Name: count, dtype: Int64


In [29]:
# Show remaining missing values.
remaining_missing = property_data.isna().sum()

display(
    remaining_missing[remaining_missing > 0]
    .sort_values(ascending=False)
    .rename("Missing count")
)

MainLevelBedrooms        124728
AssociationFee            93563
Stories                   39369
GarageSpaces              11517
LotSizeAcres               5353
LotSizeSquareFeet          5326
LotSizeArea                5302
Longitude                  2712
Latitude                   2712
StreetNumberNumeric         362
ParkingTotal                348
UnparsedAddress             294
City                        240
YearBuilt                   223
LivingArea                  164
BathroomsTotalInteger        50
PostalCode                    1
Name: Missing count, dtype: int64

In [30]:
# Mark missing location categories as Unknown.
for col in ["City", "PostalCode"]:
    property_data[col] = (
        property_data[col]
        .astype("string")
        .fillna("Unknown")
    )

print(property_data[["City", "PostalCode"]].isna().sum())

City          0
PostalCode    0
dtype: int64


In [31]:
# Inspect numerical feature ranges and missing values.
#if garage space = 0: maybe no garage
numeric_columns = property_data.select_dtypes(
    include="number"
).columns

numeric_summary = property_data[numeric_columns].describe(
    percentiles=[0.01, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

numeric_summary["Missing count"] = (
    property_data[numeric_columns].isna().sum()
)

display(
    numeric_summary[
        [
            "Missing count",
            "min",
            "1%",
            "25%",
            "50%",
            "75%",
            "95%",
            "99%",
            "max",
        ]
    ]
)

,Missing count,min,1%,25%,50%,75%,95%,99%,max
Latitude,2712,-117.472493,32.675123,33.766300,34.090283,35.110898,37.957440,39.726154,5.613037e+01
Longitude,2712,-177.646696,-122.456500,-119.299552,-118.040098,-117.265427,-116.606598,-116.241390,3.290000e+02
LivingArea,164,0.000000,728.000000,1376.000000,1804.000000,2421.000000,3809.000000,5649.000000,1.237640e+05
ParkingTotal,348,-143.000000,0.000000,2.000000,2.000000,3.000000,6.000000,11.000000,1.572000e+04
LotSizeAcres,5353,0.000000,0.041300,0.130000,0.166300,0.237600,1.135020,5.841288,7.810698e+06
YearBuilt,223,1776.000000,1910.000000,1956.000000,1976.000000,1998.000000,2022.000000,2025.000000,2.026000e+03
StreetNumberNumeric,362,0.000000,11.000000,1125.000000,3708.000000,13176.000000,39232.300000,75944.770000,7.814782e+07
BathroomsTotalInteger,50,0.000000,1.000000,2.000000,2.000000,3.000000,5.000000,6.000000,1.750000e+02
BedroomsTotal,0,0.000000,2.000000,3.000000,3.000000,4.000000,5.000000,6.000000,4.500000e+01
Stories,39369,1.000000,1.000000,1.000000,1.000000,2.000000,2.000000,2.000000,2.000000e+00


In [32]:
# Right Now, Flag clearly invalid numerical values as missing.

In [33]:
import numpy as np
import pandas as pd

# Parse sale dates before defining the training reference.
sale_dates = pd.to_datetime(sale_dates, errors="coerce")
test_start = pd.Timestamp("2026-04-01")

# Standardize numeric values.
numeric_columns = property_data.select_dtypes(
    include="number"
).columns

property_data[numeric_columns] = (
    property_data[numeric_columns]
    .replace([np.inf, -np.inf], np.nan)
)

# Recover missing lot square footage from recorded acreage.
can_convert = (
    property_data["LotSizeSquareFeet"].isna()
    & property_data["LotSizeAcres"].gt(0)
)

property_data.loc[can_convert, "LotSizeSquareFeet"] = (
    property_data.loc[can_convert, "LotSizeAcres"] * 43560
)

# Calculate screening thresholds using training months only.
training_reference = property_data.loc[
    sale_dates.lt(test_start)
]

area_columns = ["LivingArea", "LotSizeSquareFeet"]
count_columns = [
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "MainLevelBedrooms",
    "GarageSpaces",
    "ParkingTotal",
]

rules = {}
threshold_report = []

# Screen both tails of positive area measurements.
for col in area_columns:
    reference = training_reference[col]
    reference = reference.loc[reference.gt(0)]

    lower = reference.quantile(0.01)
    upper = reference.quantile(0.99)

    rules[col] = (
        property_data[col].lt(lower)
        | property_data[col].gt(upper)
    )

    threshold_report.append({
        "Feature": col,
        "Lower limit": lower,
        "Upper limit": upper,
    })

# Retain legitimate zeros in count fields.
for col in count_columns:
    reference = training_reference[col]
    reference = reference.loc[reference.ge(0)]
    upper = reference.quantile(0.99)

    rules[col] = (
        property_data[col].lt(0)
        | property_data[col].gt(upper)
    )

    threshold_report.append({
        "Feature": col,
        "Lower limit": 0,
        "Upper limit": upper,
    })

# Use a broad California bounding box.
rules["Coordinates"] = (
    (
        property_data["Latitude"].notna()
        & ~property_data["Latitude"].between(32, 42.1)
    )
    | (
        property_data["Longitude"].notna()
        & ~property_data["Longitude"].between(-124.6, -114)
    )
)

# Check consistency across related fields.
rules["Bedroom inconsistency"] = (
    property_data["MainLevelBedrooms"]
    .gt(property_data["BedroomsTotal"])
)

rules["YearBuilt"] = (
    property_data["YearBuilt"].le(0)
    | property_data["YearBuilt"].gt(sale_dates.dt.year)
)

rules["Stories"] = property_data["Stories"].le(0)
rules["Invalid sale date"] = sale_dates.isna()

# Combine rules before removing any rows.
rule_flags = pd.DataFrame(rules).fillna(False)
remove_rows = rule_flags.any(axis=1)

display(pd.DataFrame(threshold_report).round(2))
display(
    rule_flags.sum()
    .sort_values(ascending=False)
    .rename("Rows flagged per rule (may overlap)")
)

rows_before = len(property_data)

# Apply the same row selection to features and targets.
property_data = property_data.loc[~remove_rows].copy()
y = y.loc[property_data.index].copy()
sale_dates = sale_dates.loc[property_data.index].copy()

assert property_data.index.equals(y.index)
assert property_data.index.equals(sale_dates.index)

print("Rows before cleaning:", rows_before)
print("Total unique rows removed:", int(remove_rows.sum()))
print("Remaining rows:", len(property_data))
print(
    "Percentage removed:",
    round(remove_rows.mean() * 100, 2),
)

,Feature,Lower limit,Upper limit
0,LivingArea,734.0,5648.0
1,LotSizeSquareFeet,1827.0,252212.0
2,BedroomsTotal,0.0,6.0
3,BathroomsTotalInteger,0.0,6.0
4,MainLevelBedrooms,0.0,5.0
5,GarageSpaces,0.0,4.0
6,ParkingTotal,0.0,11.0


LivingArea               6306
LotSizeSquareFeet        6277
ParkingTotal             2834
BathroomsTotalInteger    2757
GarageSpaces             2206
Bedroom inconsistency    1870
BedroomsTotal            1502
MainLevelBedrooms         922
Coordinates                61
YearBuilt                  27
Stories                     0
Invalid sale date           0
Name: Rows flagged per rule (may overlap), dtype: int64

Rows before cleaning: 309710
Total unique rows removed: 19260
Remaining rows: 290450
Percentage removed: 6.22


In [34]:
# Convert sale dates and inspect their coverage.
sale_dates = pd.to_datetime(sale_dates, errors="coerce")

print("Missing or invalid sale dates:", sale_dates.isna().sum())
print("Earliest sale date:", sale_dates.min())
print("Latest sale date:", sale_dates.max())

Missing or invalid sale dates: 0
Earliest sale date: 2024-01-01 00:00:00
Latest sale date: 2026-04-30 00:00:00


In [35]:

# split data, use the most recent month of available data as the test set, 

In [36]:
# Split by sale date.
test_start = pd.Timestamp("2026-04-01")

train_mask = sale_dates.lt(test_start)
test_mask = sale_dates.ge(test_start)

X_train = property_data.loc[train_mask].copy()
X_test = property_data.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Total rows:", len(X_train) + len(X_test))

assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)
assert len(X_train) + len(X_test) == len(property_data)

Training shape: (278765, 33)
Test shape: (11685, 33)
Total rows: 290450


In [37]:
# List all retained columns.
print("Number of retained columns:", property_data.shape[1])

for number, col in enumerate(property_data.columns, start=1):
    print(f"{number}. {col}")

Number of retained columns: 33
1. Flooring
2. ViewYN
3. PoolPrivateYN
4. Latitude
5. Longitude
6. UnparsedAddress
7. PropertyType
8. LivingArea
9. AssociationFeeFrequency
10. MLSAreaMajor
11. CountyOrParish
12. AttachedGarageYN
13. ParkingTotal
14. PropertySubType
15. LotSizeAcres
16. SubdivisionName
17. YearBuilt
18. StreetNumberNumeric
19. BathroomsTotalInteger
20. City
21. BedroomsTotal
22. StateOrProvince
23. FireplaceYN
24. Stories
25. Levels
26. LotSizeArea
27. MainLevelBedrooms
28. NewConstructionYN
29. GarageSpaces
30. HighSchoolDistrict
31. PostalCode
32. AssociationFee
33. LotSizeSquareFeet


In [38]:
#impute missing values

# First batch of numeric columns to impute.
numeric_columns_to_impute = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "YearBuilt",
    "Stories",
    "MainLevelBedrooms",
    "GarageSpaces",
    "ParkingTotal",
]

# Calculate medians using training data only.
training_medians = X_train[numeric_columns_to_impute].median()

# Fill missing values using the same training medians.
X_train[numeric_columns_to_impute] = (
    X_train[numeric_columns_to_impute].fillna(training_medians)
)

X_test[numeric_columns_to_impute] = (
    X_test[numeric_columns_to_impute].fillna(training_medians)
)

# Show the values used for filling.
print("Training medians:")
print(training_medians)

# Verify remaining missing values.
print("\nRemaining missing values — training:")
print(X_train[numeric_columns_to_impute].isna().sum())

print("\nRemaining missing values — test:")
print(X_test[numeric_columns_to_impute].isna().sum())

Training medians:
LivingArea               1795.0
BedroomsTotal               3.0
BathroomsTotalInteger       2.0
YearBuilt                1975.0
Stories                     1.0
MainLevelBedrooms           3.0
GarageSpaces                2.0
ParkingTotal                2.0
dtype: float64

Remaining missing values — training:
LivingArea               0
BedroomsTotal            0
BathroomsTotalInteger    0
YearBuilt                0
Stories                  0
MainLevelBedrooms        0
GarageSpaces             0
ParkingTotal             0
dtype: int64

Remaining missing values — test:
LivingArea               0
BedroomsTotal            0
BathroomsTotalInteger    0
YearBuilt                0
Stories                  0
MainLevelBedrooms        0
GarageSpaces             0
ParkingTotal             0
dtype: int64


In [39]:
# Check remaining missing values in both datasets.
remaining_missing = pd.DataFrame({
    "Training missing": X_train.isna().sum(),
    "Test missing": X_test.isna().sum(),
})

display(
    remaining_missing.loc[
        remaining_missing.gt(0).any(axis=1)
    ].sort_values("Training missing", ascending=False)
)

,Training missing,Test missing
AssociationFee,85198,3527
LotSizeSquareFeet,4817,228
LotSizeAcres,4816,228
LotSizeArea,4812,228
Latitude,2507,70
Longitude,2507,70
StreetNumberNumeric,314,24
UnparsedAddress,253,21


In [40]:
# Retain only the standardized lot-area feature.
X_train = X_train.drop(columns=["LotSizeAcres", "LotSizeArea"])
X_test = X_test.drop(columns=["LotSizeAcres", "LotSizeArea"])

# Fill missing lot areas using the cleaned training median.
lot_median = X_train["LotSizeSquareFeet"].median()

X_train["LotSizeSquareFeet"] = (
    X_train["LotSizeSquareFeet"].fillna(lot_median)
)
X_test["LotSizeSquareFeet"] = (
    X_test["LotSizeSquareFeet"].fillna(lot_median)
)

print("Training lot-area median:", lot_median)
print("Training missing:", X_train["LotSizeSquareFeet"].isna().sum())
print("Test missing:", X_test["LotSizeSquareFeet"].isna().sum())

Training lot-area median: 7200.0
Training missing: 0
Test missing: 0


In [41]:
# Remove address fields from model inputs.
address_columns = ["UnparsedAddress", "StreetNumberNumeric"]

X_train = X_train.drop(columns=address_columns)
X_test = X_test.drop(columns=address_columns)

print("Training columns:", X_train.shape[1])
print("Test columns:", X_test.shape[1])

Training columns: 29
Test columns: 29


In [42]:
# Calculate coordinate medians by postal code from training data.
coordinate_columns = ["Latitude", "Longitude"]

postal_coordinate_medians = (
    X_train.loc[X_train["PostalCode"].ne("Unknown")]
    .groupby("PostalCode")[coordinate_columns]
    .median()
)

# Fill missing coordinates using the same training lookup.
for dataset in [X_train, X_test]:
    for col in coordinate_columns:
        estimated_coordinates = dataset["PostalCode"].map(
            postal_coordinate_medians[col]
        )

        dataset[col] = dataset[col].fillna(estimated_coordinates)

# Check remaining missing coordinates.
print("Training missing coordinates:")
print(X_train[coordinate_columns].isna().sum())

print("\nTest missing coordinates:")
print(X_test[coordinate_columns].isna().sum())

Training missing coordinates:
Latitude     55
Longitude    55
dtype: int64

Test missing coordinates:
Latitude     1
Longitude    1
dtype: int64


In [43]:
# Keep rows with both coordinates available.
train_valid = X_train[["Latitude", "Longitude"]].notna().all(axis=1)
test_valid = X_test[["Latitude", "Longitude"]].notna().all(axis=1)

# Show how many rows will be removed.
print("Training rows removed:", (~train_valid).sum())
print("Test rows removed:", (~test_valid).sum())

# Keep the same rows in features and targets.
X_train = X_train.loc[train_valid]
y_train = y_train.loc[train_valid]

X_test = X_test.loc[test_valid]
y_test = y_test.loc[test_valid]

print("Remaining training rows:", len(X_train))
print("Remaining test rows:", len(X_test))

Training rows removed: 55
Test rows removed: 1
Remaining training rows: 278710
Remaining test rows: 11684


In [44]:
# Summarize HOA fees by payment frequency in training data.
display(
    X_train.groupby("AssociationFeeFrequency")["AssociationFee"]
    .agg(["size", "count", "median"])
    .rename(columns={
        "size": "Total rows",
        "count": "Rows with fee",
        "median": "Median fee",
    })
)

,Total rows,Rows with fee,Median fee
AssociationFeeFrequency,,,
Annually,5601,5576,380.0
Monthly,60884,60863,170.0
Quarterly,2534,2533,255.0
SemiAnnually,329,329,179.0
Unknown,209362,124253,0.0


In [45]:
# Drop HOA fields because 74.9% of training records have an unknown payment frequency,
# preventing reliable conversion of fees to a consistent monthly amount. 

In [46]:
# Drop HOA fields if they are still present.
hoa_columns = ["AssociationFee", "AssociationFeeFrequency"]

X_train = X_train.drop(columns=hoa_columns, errors="ignore")
X_test = X_test.drop(columns=hoa_columns, errors="ignore")

print("Training columns:", X_train.shape[1])
print("Test columns:", X_test.shape[1])

Training columns: 27
Test columns: 27


In [47]:
# Convert categorical fields to numeric

In [48]:
# Identify categorical columns.
categorical_columns = X_train.select_dtypes(
    include=["object", "string", "category", "bool"]
).columns.tolist()

# Count distinct values in each categorical column.
display(
    X_train[categorical_columns]
    .nunique()
    .sort_values(ascending=False)
    .rename("Number of categories")
)

SubdivisionName       11196
PostalCode             2929
MLSAreaMajor           1048
City                   1046
HighSchoolDistrict      434
Flooring                309
CountyOrParish           60
Levels                   18
ViewYN                    3
PoolPrivateYN             3
AttachedGarageYN          3
FireplaceYN               3
NewConstructionYN         3
PropertyType              1
PropertySubType           1
StateOrProvince           1
Name: Number of categories, dtype: int64

In [49]:
# Inspect state values in both datasets.
print("Training state values:")
print(X_train["StateOrProvince"].value_counts(dropna=False))

print("\nTest state values:")
print(X_test["StateOrProvince"].value_counts(dropna=False))

Training state values:
StateOrProvince
CA    278710
Name: count, dtype: int64

Test state values:
StateOrProvince
CA    11684
Name: count, dtype: int64


In [50]:
# Remove constant scope fields from model inputs.
scope_columns = [
    "StateOrProvince",
    "PropertyType",
    "PropertySubType",
]

X_train = X_train.drop(columns=scope_columns, errors="ignore")
X_test = X_test.drop(columns=scope_columns, errors="ignore")

print("Training columns:", X_train.shape[1])
print("Test columns:", X_test.shape[1])

Training columns: 24
Test columns: 24


In [51]:
# Identify remaining categorical columns.
categorical_columns = X_train.select_dtypes(
    include=["object", "string", "category", "bool"]
).columns.tolist()

# Standardize category formatting in both datasets.
for dataset in [X_train, X_test]:
    for col in categorical_columns:
        dataset[col] = (
            dataset[col]
            .astype("string")
            .str.strip()
            .str.lower()
            .replace("", pd.NA)
            .fillna("unknown")
        )

# Review category counts after standardization.
display(
    X_train[categorical_columns]
    .nunique()
    .sort_values(ascending=False)
    .rename("Number of categories")
)

SubdivisionName       10797
PostalCode             2929
MLSAreaMajor           1048
City                   1041
HighSchoolDistrict      434
Flooring                309
CountyOrParish           60
Levels                   18
ViewYN                    3
PoolPrivateYN             3
AttachedGarageYN          3
FireplaceYN               3
NewConstructionYN         3
Name: Number of categories, dtype: int64

In [52]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Identify numeric columns.
numeric_columns = [
    col for col in X_train.columns
    if col not in categorical_columns
]

# Configure categorical encoding and retain numeric features.
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True,
            ),
            categorical_columns,
        ),
        ("numeric", "passthrough", numeric_columns),
    ],
    sparse_threshold=1.0,
)

# Learn categories from training data only.
X_train_encoded = preprocessor.fit_transform(X_train)

# Apply the same encoding to test data.
X_test_encoded = preprocessor.transform(X_test)

print("Encoded training shape:", X_train_encoded.shape)
print("Encoded test shape:", X_test_encoded.shape)

Encoded training shape: (278710, 16662)
Encoded test shape: (11684, 16662)


In [53]:
from pathlib import Path

# Use the existing project folder.
OUTPUT_DIR = (
    Path.home()
    / "Downloads"
    / "IDX-Exchange-Data-Science"
)

# Check the folder and data alignment.
assert OUTPUT_DIR.is_dir(), f"Project folder not found: {OUTPUT_DIR}"
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)
assert X_train.columns.equals(X_test.columns)

# Include the target and sale date for export only.
train_export = X_train.assign(
    ClosePrice=y_train,
    CloseDate=sale_dates.loc[X_train.index],
)

test_export = X_test.assign(
    ClosePrice=y_test,
    CloseDate=sale_dates.loc[X_test.index],
)

# Save cleaned training and test CSV files.
train_path = OUTPUT_DIR / "single_family_train_cleaned.csv"
test_path = OUTPUT_DIR / "single_family_test_cleaned.csv"

train_export.to_csv(train_path, index=False)
test_export.to_csv(test_path, index=False)

print("Training CSV saved:", train_path)
print("Training shape:", train_export.shape)

print("\nTest CSV saved:", test_path)
print("Test shape:", test_export.shape)

Training CSV saved: /Users/jasmineyuan/Downloads/IDX-Exchange-Data-Science/single_family_train_cleaned.csv
Training shape: (278710, 26)

Test CSV saved: /Users/jasmineyuan/Downloads/IDX-Exchange-Data-Science/single_family_test_cleaned.csv
Test shape: (11684, 26)


In [54]:
import joblib

# Verify that encoded features match the target lengths.
assert X_train_encoded.shape[0] == len(y_train)
assert X_test_encoded.shape[0] == len(y_test)

# Save encoded features, targets, and encoding rules together.
modeling_data = {
    "X_train": X_train_encoded,
    "X_test": X_test_encoded,
    "y_train": y_train.to_numpy(),
    "y_test": y_test.to_numpy(),
    "preprocessor": preprocessor,
}

joblib.dump(
    modeling_data,
    OUTPUT_DIR / "week3_modeling_data.joblib",
    compress=3,
)

print("Modeling data saved.")

Modeling data saved.
